# FZ Variable Syntax & Formulas

fz supports a rich template language for input files:

| Feature | Syntax | Example |
|---------|--------|---------|
| Simple variable | `$name` | `$x` |
| Delimited variable | `${name}` | `${x}` |
| Variable with default | `${name~value}` | `${x~3.14}` |
| Formula | `@{expression}` | `@{x * 2 + 1}` |
| Context code line | `#@ code` | `#@ import math` |
| Static constant | `#@: NAME = value` | `#@: PI = 3.14159` |
| Legacy Java syntax | `?(name)` | `?(x)` |

This notebook exhaustively tests each feature.


In [1]:
import json
from pathlib import Path
import fz

fz.set_log_level("WARNING")
WORK = Path("work_02_syntax")
WORK.mkdir(exist_ok=True)

MODEL = {
    "varprefix": "$",
    "formulaprefix": "@",
    "delim": "{}",
    "commentline": "#",
    "output": {},
}


In [2]:
def read_compiled(out_dir, filename):
    """Read compiled file – handles both direct and subdirectory layouts."""
    direct = Path(out_dir) / filename
    if direct.exists():
        return direct.read_text()
    subdirs = sorted(d for d in Path(out_dir).iterdir()
                     if d.is_dir() and not d.name.startswith("."))
    if subdirs:
        return (subdirs[0] / filename).read_text()
    raise FileNotFoundError(f"{filename} not found in {out_dir}")

def case_dir(out_dir):
    """Return the first case subdirectory (or out_dir itself for defaults)."""
    p = Path(out_dir)
    subdirs = sorted(d for d in p.iterdir()
                     if d.is_dir() and not d.name.startswith("."))
    return subdirs[0] if subdirs else p


## 1 · Simple variables: `$name` and `${name}`

In [3]:
tmpl = WORK / "simple.in"
tmpl.write_text(
    "radius = $r\n"
    "diameter = ${d}\n"
    "area_approx = 3.14159 * $r * $r\n"
)
vars_ = fz.fzi(str(tmpl), MODEL)
print("Detected:", vars_)

out = WORK / "simple_out"
fz.fzc(str(tmpl), {"r": 5.0, "d": 10.0}, MODEL, output_dir=str(out))
print("\nCompiled:")
print(read_compiled(out, "simple.in"))


Detected: {'d': None, 'r': None}

Compiled:
radius = 5.0
diameter = 10.0
area_approx = 3.14159 * 5.0 * 5.0



## 2 · Default values: `${name~default}`

In [4]:
tmpl = WORK / "defaults.in"
tmpl.write_text(
    "speed     = ${v~100.0}   # default 100 m/s\n"
    "mass      = ${m~1.0}     # default 1 kg\n"
    "frequency = ${f~50}      # default 50 Hz\n"
)
vars_ = fz.fzi(str(tmpl), MODEL)
print("Variables with defaults:", vars_)

# Compile WITHOUT overriding — defaults are used
out_def = WORK / "defaults_out_default"
fz.fzc(str(tmpl), {}, MODEL, output_dir=str(out_def))
print("\nCompiled with no overrides (uses defaults):")
print(read_compiled(out_def, "defaults.in"))

# Compile overriding some
out_ov = WORK / "defaults_out_override"
fz.fzc(str(tmpl), {"v": 200.0, "f": 60}, MODEL, output_dir=str(out_ov))
print("Compiled with v=200, f=60  (m stays at default 1.0):")
print(read_compiled(out_ov, "defaults.in"))


Variables with defaults: {'f': 50, 'm': 1.0, 'v': 100.0}

Compiled with no overrides (uses defaults):
speed     = 100.0   # default 100 m/s
mass      = 1.0     # default 1 kg
frequency = 50      # default 50 Hz

Compiled with v=200, f=60  (m stays at default 1.0):
speed     = 200.0   # default 100 m/s
mass      = 1.0     # default 1 kg
frequency = 60      # default 50 Hz



## 3 · Formula evaluation: `@{expression}`

Formulas are evaluated with the Python interpreter at compile time.

In [5]:
tmpl = WORK / "formulas.in"
tmpl.write_text(
    "# Basic formula\n"
    "x = ${x~2.0}\n"
    "y = ${y~3.0}\n"
    "sum_sq  = @{$x**2 + $y**2}\n"
    "product = @{$x * $y}\n"
    "dist    = @{($x**2 + $y**2)**0.5}\n"
)
vars_ = fz.fzi(str(tmpl), MODEL)
print("Variables:", vars_)

out = WORK / "formulas_out"
fz.fzc(str(tmpl), {"x": 3.0, "y": 4.0}, MODEL, output_dir=str(out))
print("\nCompiled (expected: sum_sq=25, product=12, dist=5):")
print(read_compiled(out, "formulas.in"))


Variables: {'x': 2.0, 'y': 3.0, 'x**2 + y**2': 13.0, 'x * y': 6.0, '(x**2 + y**2)**0.5': 3.605551275463989}

Compiled (expected: sum_sq=25, product=12, dist=5):
# Basic formula
x = 3.0
y = 4.0
sum_sq  = 25.0
product = 12.0
dist    = 5.0



## 4 · Context code lines: `#@ code`

Lines starting with `#@` define helper functions or imports available inside `@{}` formulas.

In [6]:
tmpl = WORK / "context.in"
tmpl.write_text(
    "#@ import math\n"
    "#@ def celsius_to_kelvin(c):\n"
    "#@     return c + 273.15\n"
    "#@ def beam_deflection(F, L, E, I):\n"
    "#@     return F * L**3 / (48 * E * I)\n"
    "\n"
    "T_celsius = ${T_celsius~20.0}\n"
    "temperature_K = @{celsius_to_kelvin($T_celsius)}\n"
    "force     = ${F~1000.0}\n"
    "length    = ${L~2.0}\n"
    "youngs    = ${E~200e9}\n"
    "inertia   = ${I~8.333e-6}\n"
    "deflection_mm = @{beam_deflection($F, $L, $E, $I) * 1000}\n"
)
vars_ = fz.fzi(str(tmpl), MODEL)
print("Variables:", vars_)

out = WORK / "context_out"
fz.fzc(str(tmpl), {"T_celsius": 25.0, "F": 5000.0, "L": 3.0, "E": 200e9, "I": 8.333e-6},
       MODEL, output_dir=str(out))
print("\nCompiled:")
print(read_compiled(out, "context.in"))


Variables: {'E': 200000000000.0, 'F': 1000.0, 'I': 8.333e-06, 'L': 2.0, 'T_celsius': 20.0, 'celsius_to_kelvin(T_celsius)': None, 'beam_deflection(F, L, E, I) * 1000': None}

Compiled:
#@ import math
#@ def celsius_to_kelvin(c):
#@     return c + 273.15
#@ def beam_deflection(F, L, E, I):
#@     return F * L**3 / (48 * E * I)

T_celsius = 25.0
temperature_K = 298.15
force     = 5000.0
length    = 3.0
youngs    = 200000000000.0
inertia   = 8.333e-06
deflection_mm = 1.687567502700108



## 5 · Static constants: `#@: NAME = value`

Static constants are defined once and reused in formulas.  They are **not** treated as input variables.

In [7]:
tmpl = WORK / "static.in"
tmpl.write_text(
    "#@: G = 9.81\n"
    "#@: PI = 3.14159265\n"
    "\n"
    "# Pendulum period: T = 2*PI*sqrt(L/G)\n"
    "L = ${L~1.0}\n"
    "period = @{2 * PI * (L / G) ** 0.5}\n"
    "\n"
    "# Kinetic energy: Ek = 0.5*m*v^2\n"
    "m = ${m~1.0}\n"
    "v = ${v~10.0}\n"
    "Ek = @{0.5 * m * v**2}\n"
)
vars_ = fz.fzi(str(tmpl), MODEL)
print("Input variables (G and PI not listed — they are static constants):")
print(vars_)

out = WORK / "static_out"
fz.fzc(str(tmpl), {"L": 0.5, "m": 2.0, "v": 5.0}, MODEL, output_dir=str(out))
print("\nCompiled:")
print(read_compiled(out, "static.in"))


Input variables (G and PI not listed — they are static constants):
{'G': '9.81', 'PI': '3.14159265', 'L': 1.0, 'm': 1.0, 'v': 10.0, '2 * PI * (L / G) ** 0.5': 2.006066678418382, '0.5 * m * v**2': 50.0}

Compiled:
#@: G = 9.81
#@: PI = 3.14159265

# Pendulum period: T = 2*PI*sqrt(L/G)
L = 0.5
period = 1.418503351822011

# Kinetic energy: Ek = 0.5*m*v^2
m = 2.0
v = 5.0
Ek = 25.0



## 6 · Legacy Java/Funz syntax: `?(name)`

In [8]:
tmpl = WORK / "legacy.in"
tmpl.write_text(
    "# Legacy Funz Java variable syntax — fz auto-converts it\n"
    "x = ?(x)\n"
    "y = ?(y)\n"
    "z = ?(z~0.0)\n"
)
vars_ = fz.fzi(str(tmpl), MODEL)
print("Variables from legacy syntax:", vars_)

out = WORK / "legacy_out"
fz.fzc(str(tmpl), {"x": 1.0, "y": 2.0}, MODEL, output_dir=str(out))
print("\nCompiled:")
print(read_compiled(out, "legacy.in"))


Variables from legacy syntax: {}
  ⚠️  Warning: The following input variables are not found in input files: x, y

Compiled:
# Legacy Funz Java variable syntax — fz auto-converts it
x = ?(x)
y = ?(y)
z = ?(z~0.0)



## 7 · Custom model key aliases

Model keys accept multiple aliases: `varprefix`/`varchar`/`var_prefix`/`var_char`.

In [9]:
for var_key in ("varprefix", "varchar", "var_prefix", "var_char"):
    m = {var_key: "$", "formulaprefix": "@", "delim": "{}", "commentline": "#", "output": {}}
    tmpl2 = WORK / f"alias_{var_key}.in"
    tmpl2.write_text("x = ${x~1.0}\n")
    v = fz.fzi(str(tmpl2), m)
    print(f"  {var_key:15s} → {v}")


  varprefix       → {'x': 1.0}
  varchar         → {'x': 1.0}
  var_prefix      → {'x': 1.0}
  var_char        → {'x': 1.0}


## 8 · Different delimiter styles

In [10]:
configs = [
    ("()",  "x = $(x~1.0)"),
    ("{}",  "x = ${x~1.0}"),
    ("[]",  "x = $[x~1.0]"),
    ("<>",  "x = $<x~1.0>"),
]
for delim, line in configs:
    m = {"varprefix": "$", "delim": delim, "commentline": "#", "output": {}}
    f = WORK / f"delim_{delim[0]}.in"
    f.write_text(line + "\n")
    v = fz.fzi(str(f), m)
    print(f"  delim={delim!r:5s}  template={line!r:25s}  → {v}")


  delim='()'   template='x = $(x~1.0)'             → {'x': 1.0}
  delim='{}'   template='x = ${x~1.0}'             → {'x': 1.0}
  delim='[]'   template='x = $[x~1.0]'             → {'x': 1.0}
  delim='<>'   template='x = $<x~1.0>'             → {'x': 1.0}


## 9 · Complex formulas: chained context + multi-var

In [11]:
tmpl = WORK / "complex.in"
tmpl.write_text(
    "#@: PI = 3.14159265358979\n"
    "#@: G = 9.81\n"
    "#@ def sphere_volume(r):\n"
    "#@     return (4/3) * PI * r**3\n"
    "#@ def projectile_range(v0, angle_deg):\n"
    "#@     import math\n"
    "#@     a = math.radians(angle_deg)\n"
    "#@     return v0**2 * math.sin(2*a) / G\n"
    "radius   = ${r~1.0}\n"
    "v0       = ${v0~20.0}\n"
    "angle    = ${angle~45.0}\n"
    "volume   = @{sphere_volume($r)}\n"
    "range_m  = @{projectile_range($v0, $angle)}\n"
)
vars_ = fz.fzi(str(tmpl), MODEL)
print("Variables:", vars_)

out = WORK / "complex_out"
fz.fzc(str(tmpl), {"r": 2.5, "v0": 50.0, "angle": 30.0}, MODEL, output_dir=str(out))
print("\nCompiled  (vol ≈ 65.45 m³; range ≈ 220.4 m):")
print(read_compiled(out, "complex.in"))


Variables: {'PI': '3.14159265358979', 'G': '9.81', 'angle': 45.0, 'r': 1.0, 'v0': 20.0, 'sphere_volume(r)': None, 'projectile_range(v0, angle)': None}

Compiled  (vol ≈ 65.45 m³; range ≈ 220.4 m):
#@: PI = 3.14159265358979
#@: G = 9.81
#@ def sphere_volume(r):
#@     return (4/3) * PI * r**3
#@ def projectile_range(v0, angle_deg):
#@     import math
#@     a = math.radians(angle_deg)
#@     return v0**2 * math.sin(2*a) / G
radius   = 2.5
v0       = 50.0
angle    = 30.0
volume   = 65.44984694978729
range_m  = 220.69964418563674



## 10 · `fzr` with formula-bearing templates

In [12]:
SIM = WORK / "models" / "sim.py"
(WORK / "models").mkdir(exist_ok=True)
SIM.write_text('''#!/usr/bin/env python3
params = {}
with open("complex.in") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line and not line.startswith("@"):
            k, v = line.split("=", 1)
            try:
                params[k.strip()] = float(v.strip())
            except ValueError:
                pass
with open("output.txt", "w") as fh:
    for k, v in params.items():
        fh.write(f"{k} = {v}\\n")
''')

MODEL2 = {
    "varprefix": "$", "formulaprefix": "@", "delim": "{}",
    "commentline": "#",
    "output": {
        "volume":   "grep '^volume' output.txt | awk '{print $3}'",
        "range_m":  "grep '^range_m' output.txt | awk '{print $3}'",
    }
}

import pandas as pd
CALC = f"sh://python3 {SIM.resolve()}"
df = fz.fzr(
    str(tmpl),
    {"r": [1.0, 2.0, 3.0], "v0": [20.0, 40.0], "angle": [30.0, 45.0, 60.0]},
    MODEL2,
    results_dir=str(WORK / "results"),
    calculators=[CALC],
)
df["volume"]  = df["volume"].astype(float)
df["range_m"] = df["range_m"].astype(float)
print(df[["r", "v0", "angle", "volume", "range_m"]].to_string(index=False))


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/18) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/18) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/18) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/18) ETA: ...

◢ [██>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   5% (1/18) ETA: 9s

...

◢ [█████████████████████████████████████>░░]  94% (17/18) ETA: 0s

◣ [█████████████████████████████████████>░░]  94% (17/18) ETA: 0s

◤ [█████████████████████████████████████>░░]  94% (17/18) ETA: 0s

  [████████████████████████████████████████] 100% (18/18) Total: 9s

  r   v0  angle     volume    range_m
1.0 20.0   30.0   4.188790  35.311943
1.0 20.0   45.0   4.188790  40.774720
1.0 20.0   60.0   4.188790  35.311943
1.0 40.0   30.0   4.188790 141.247772
1.0 40.0   45.0   4.188790 163.098879
1.0 40.0   60.0   4.188790 141.247772
2.0 20.0   30.0  33.510322  35.311943
2.0 20.0   45.0  33.510322  40.774720
2.0 20.0   60.0  33.510322  35.311943
2.0 40.0   30.0  33.510322 141.247772
2.0 40.0   45.0  33.510322 163.098879
2.0 40.0   60.0  33.510322 141.247772
3.0 20.0   30.0 113.097336  35.311943
3.0 20.0   45.0 113.097336  40.774720
3.0 20.0   60.0 113.097336  35.311943
3.0 40.0   30.0 113.097336 141.247772
3.0 40.0   45.0 113.097336 163.098879
3.0 40.0   60.0 113.097336 141.247772
